 Project : Building Generative Pretrained Transformer ( Decoder Only)

**Course:** TECH 77 — Modern AI Architectures: From RNNs to Chatbots  
**Format:** Google Colab  
**Estimated Time:** 6- 7 hours

## Project Overview

In this Project, We will explore the architecture of self-attention based transformer decoder.

We will:

- Design Input embedding layer
- Add positional encoding to the token embeddings
- Parse through Masked multi head attention
- Add residual and normalize the layer
- Parse it through Feed Forward Neural Network
- Add residual and normalize the layer
- Parse it through linear layer and through softmax for output probabilities
  

This notebook focuses on the Decoder only Architecture and does not build a complete encoder–decoder Transformer.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input,
    Embedding,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Dropout,
    Add,
    LayerNormalization,
    Dense
)
from tensorflow.keras.models import Model
import string
from collections import Counter


np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [ ]:
#Loading from the IMDB corpus to train the word to integer relation on IMDB Dataset

vocab_size_imdb = 5000   # capping the vocabulary to the 5,000 most frequent words
max_len = 50              # fixed sequence length every review is padded/truncated to

# Dropping the sentiment labels (y_train/y_test) -- we only need the review text
# itself to build a vocabulary and train a language model, not sentiment.
(x_train, _), (x_test, _) = imdb.load_data(num_words=vocab_size_imdb)

word_index = imdb.get_word_index()
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i, "?") for i in encoded_review])

def simple_tokenize(text):
    word_lower_case = text.lower()
    clear_punctuation = ''.join(ch for ch in word_lower_case if ch not in string.punctuation)
    split_text_words = clear_punctuation.split()
    special_tokens = {"start", "unk", "pad", "unused"}
    return [w for w in split_text_words if w not in special_tokens]

# decode_review / simple_tokenize are kept for inspection/debugging only --
# they are NOT used to build the training data below, since we're using
# IMDB's original integer encoding directly. Uncomment to sanity-check a review:

# sample_text = decode_review(x_train[0])
# tokens = simple_tokenize(sample_text)
# print(tokens[:40])

# NOTE: max_len is 50 here for embedding purposes. If/when this moves to
# language-model training (predict-the-next-word), re-pad with
# maxlen=max_len + 1 instead, so there's room to split each sequence into
# input = tokens[:-1] / target = tokens[1:].
x_train_lm_padded = pad_sequences(x_train, maxlen=max_len + 1, padding="post", truncating="post")
x_test_lm_padded = pad_sequences(x_test, maxlen=max_len + 1, padding="post", truncating="post")

# Split each sequence into input (first max_len tokens) and target (shifted by one)
x_train_input = x_train_lm_padded[:, :-1]   # shape (num_reviews, max_len) -- model input
x_train_target = x_train_lm_padded[:, 1:]   # shape (num_reviews, max_len) -- next-token labels

x_test_input = x_test_lm_padded[:, :-1]
x_test_target = x_test_lm_padded[:, 1:]

print("x_train_input shape:", x_train_input.shape)     # (num_reviews, 50)
print("x_train_target shape:", x_train_target.shape)   # (num_reviews, 50)

## In this block we have loaded the data from IMDB dataset for training and padded each review to be of the samelength.

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
x_train_input shape: (25000, 50)
x_train_target shape: (25000, 50)


In [ ]:
#Embedding layer to convert the integer representing the words into word to vector for classifying them with respect to the correct class.
embedding_dim = 64
inputs = Input(shape=(max_len,))
embedding_layer = Embedding(input_dim=vocab_size_imdb, output_dim=embedding_dim)
embedded_output = embedding_layer(inputs)

embedding_model = Model(inputs=inputs, outputs=embedded_output)

# Run a small batch through it to confirm shapes
sample_embeddings = embedding_model.predict(x_train_input[:5], verbose=0)

print("x_train_input shape:", x_train_input[:5].shape)   # (5, 50)  -- integers
print("sample_embeddings shape:", sample_embeddings.shape)   # (5, 50, 64)        -- vector

x_train_input shape: (5, 50)
sample_embeddings shape: (5, 50, 64)


In [ ]:
# Adding positional encoding to the word2vector intialization above .

# Reused from Assignment 3
def create_positional_encoding(sequence_length, embedding_dim):
    position = np.arange(sequence_length)[:, np.newaxis]
    dimension = np.arange(embedding_dim)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (dimension // 2)) / np.float32(embedding_dim)
    )
    angle_radians = position * angle_rates

    positional_encoding = np.zeros((sequence_length, embedding_dim))
    positional_encoding[:, 0::2] = np.sin(angle_radians[:, 0::2])
    positional_encoding[:, 1::2] = np.cos(angle_radians[:, 1::2])
    return positional_encoding

# Replaces the learned position_embedding_layer / positions / position_embeddings
positional_encoding_matrix = create_positional_encoding(max_len, embedding_dim)
position_embeddings = tf.constant(positional_encoding_matrix, dtype=tf.float32)

combined_embeddings = embedded_output + position_embeddings

embedding_pos_model = Model(inputs=inputs, outputs=combined_embeddings)

sample_combined = embedding_pos_model.predict(x_train_input[:5], verbose=0)

print("position_embeddings shape:", position_embeddings.shape)
print("sample_combined shape:", sample_combined.shape)

position_embeddings shape: (50, 64)
sample_combined shape: (5, 50, 64)


# Below is the code done manually to sketch each layer for clear understanding but this would be optimized into a function later for easier access.
#Constructing masked multi-head attention function

num_heads = 4
key_dim = 16

masked_attention_layer = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
attention_output = masked_attention_layer(combined_embeddings, combined_embeddings, use_causal_mask=True)

print("combined_embeddings shape:", combined_embeddings.shape)
print("attention_output shape:", attention_output.shape)

# Adding the residual and layer normalization
residual_output = Add()([combined_embeddings, attention_output])
normalized_output = LayerNormalization(epsilon=1e-6)(residual_output)

print("combined_embeddings shape:", combined_embeddings.shape)
print("attention_output shape:", attention_output.shape)
print("residual_output shape:", residual_output.shape)
print("normalized_output shape:", normalized_output.shape)


# Adding the FeedForward Network
ff_dim = 128  # feed-forward inner dimension (2x embedding_dim here)

ffn_hidden = Dense(ff_dim, activation="relu")(normalized_output)
ffn_output = Dense(embedding_dim)(ffn_hidden)

print("normalized_output shape:", normalized_output.shape)
print("ffn_hidden shape:", ffn_hidden.shape)
print("ffn_output shape:", ffn_output.shape)

# Adding the residual and layer normalization
ffn_residual_output = Add()([normalized_output, ffn_output])
decoder_block_output = LayerNormalization(epsilon=1e-6)(ffn_residual_output)
print("normalized_output shape:", normalized_output.shape)
print("ffn_output shape:", ffn_output.shape)
print("ffn_residual_output shape:", ffn_residual_output.shape)
print("decoder_block_output shape:", decoder_block_output.shape)

In [ ]:
def transformer_decoder_block(x, embedding_dim, num_heads, key_dim, ff_dim):
    # Masked self-attention
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x, use_causal_mask=True)

    # First residual connection + Layer Normalization
    x = Add()([x, attention_output])
    x = LayerNormalization(epsilon=1e-6)(x)

    # Feed-forward network
    ffn_hidden = Dense(ff_dim, activation="relu")(x)
    ffn_output = Dense(embedding_dim)(ffn_hidden)

    # Second residual connection + Layer Normalization
    x = Add()([x, ffn_output])
    x = LayerNormalization(epsilon=1e-6)(x)

    return x

In [ ]:
num_heads = 4
key_dim = 16
ff_dim = 128

decoder_block_1_output = transformer_decoder_block(combined_embeddings, embedding_dim, num_heads, key_dim, ff_dim)
decoder_block_2_output = transformer_decoder_block(decoder_block_1_output, embedding_dim, num_heads, key_dim, ff_dim)

print("combined_embeddings shape:", combined_embeddings.shape)
print("decoder_block_1_output shape:", decoder_block_1_output.shape)
print("decoder_block_2_output shape:", decoder_block_2_output.shape)

combined_embeddings shape: (None, 50, 64)
decoder_block_1_output shape: (None, 50, 64)
decoder_block_2_output shape: (None, 50, 64)


In [ ]:
#Adding output layer
outputs = Dense(vocab_size_imdb, activation="softmax")(decoder_block_2_output)

print("decoder_block_2_output shape:", decoder_block_2_output.shape)
print("outputs shape:", outputs.shape)


decoder_block_2_output shape: (None, 50, 64)
outputs shape: (None, 50, 5000)


In [ ]:
#Model for training.
model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 64)    │    320,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 64)    │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 64)    │     16,640 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 50, 64)    │          0 │ add[0][0],        │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 50, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50, 64)    │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 50, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 64)    │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 50, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 64)    │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 50, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 50, 64)    │      8,256 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 50, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 64)    │        128 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 50, 5000)  │    325,000 │ layer_normalizat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 711,944 (2.72 MB)

 Trainable params: 711,944 (2.72 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
train_sample_weight = (x_train_target != 0).astype("float32")
history = model.fit(
    x_train_input,
    x_train_target,
    sample_weight=train_sample_weight,
    epochs=1000,                  # upper bound only — early stopping will cut this short
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/1000
313/313 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.1117 - loss: 5.9601 - val_accuracy: 0.1577 - val_loss: 5.3139
Epoch 2/1000
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1705 - loss: 5.1125 - val_accuracy: 0.1780 - val_loss: 5.0059
Epoch 3/1000
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1861 - loss: 4.8834 - val_accuracy: 0.1874 - val_loss: 4.8734
Epoch 4/1000
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.1941 - loss: 4.7482 - val_accuracy: 0.1928 - val_loss: 4.7927
Epoch 5/1000
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.1997 - loss: 4.6505 - val_accuracy: 0.1960 - val_loss: 4.7423
Epoch 6/1000
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.2039 - loss: 4.5770 - val_accuracy: 0.1977 - val_loss: 4.7135
Epoch 7/1000
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.2072 - loss: 4.5196 - val_accuracy: 0.1998 - val_loss: 4.6911
Epoch 8/1000
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.2099 - loss: 

In [ ]:
model.save("gpt_model.keras")

from google.colab import files
files.download("gpt_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:


def encode_text(text, word_index, vocab_size):
    tokens = text.lower().split()
    encoded = [1]  # <START> token, matching how IMDB reviews are encoded
    for token in tokens:
        index = word_index.get(token, 2 - 3) + 3  # unseen words -> <UNK>
        encoded.append(index if index < vocab_size else 2)
    return encoded


def sample_with_temperature(probabilities, temperature=1.0):
    probabilities = np.asarray(probabilities).astype("float64")
    if temperature <= 0:
        return int(np.argmax(probabilities))
    log_probs = np.log(probabilities + 1e-9) / temperature
    scaled = np.exp(log_probs)
    scaled = scaled / np.sum(scaled)
    return int(np.random.choice(len(scaled), p=scaled))


def generate_text(model, seed_text, num_words_to_generate=30, temperature=0.8):
    generated_tokens = encode_text(seed_text, word_index, vocab_size_imdb)

    for _ in range(num_words_to_generate):
        context = generated_tokens[-max_len:]
        pad_amount = max_len - len(context)
        model_input = np.array([[0] * pad_amount + context])

        predictions = model.predict(model_input, verbose=0)[0]
        last_real_position = pad_amount + len(context) - 1
        next_token_probs = predictions[last_real_position]

        next_token = sample_with_temperature(next_token_probs, temperature)
        generated_tokens.append(next_token)

    words = [reverse_word_index.get(token, "") for token in generated_tokens]
    return " ".join(word for word in words if word)

reverse_word_index[1] = ""
reverse_word_index[2] = ""   # hide <UNK> as well
reverse_word_index[3] = ""
#seed_text = "The movie was"
experiments = [
    ("this movie was", 0.3),
    ("this movie was", 1.2),
    ("i really enjoyed", 0.8),
    ("the acting was", 0.8),
    ("the plot was", 0.8),
]

for seed, temp in experiments:
    result = generate_text(model, seed, num_words_to_generate=30, temperature=temp)
    print(f"Seed: '{seed}' | Temperature: {temp}")
    print("\nGenerated text:")
    print(result)
    print("---")

#generated = generate_text(model, seed_text, num_words_to_generate=30, temperature=0.8)
#print("Seed:", seed_text)
#print("\nGenerated text:")
#print(generated)

Seed: 'this movie was' | Temperature: 0.3

Generated text:
this movie was pretty good and the
---
Seed: 'this movie was' | Temperature: 1.2

Generated text:
this movie was spot never a car and annoying music hall i have to admit that the movie terrible and sweet soundtrack and the 1960s only funny exception musicals measure here you are
---
Seed: 'i really enjoyed' | Temperature: 0.8

Generated text:
i really enjoyed that movie in history is of is an entertaining disaster to the middle of sam van had been planned to see stories of
---
Seed: 'the acting was' | Temperature: 0.8

Generated text:
the acting was predictable and very slow so unbelievable the story follows his to and to state prison for a when a student show using his
---
Seed: 'the plot was' | Temperature: 0.8

Generated text:
the plot was entertaining and surreal also and good reactions to the movie ended up no sense of the itself was absent cliché that would expect much to be
---
